# Overall Bias and Noise Comparisons Across Models and Packages

A one-figure-per-metric summary of the random-restart behaviour of every model we
study, for every package setup:

| | Setups (columns) |
|---|---|
| **PyMC** | default · 100 MC · Adam · Adam + 100 MC |
| **TFP\*\*** | default · 100 MC |
| **NumPyro** | default · 100 MC |
| **Stan** | default · 100 MC (both with `tol_rel_obj=1e-6` and 10k output draws) |

\*PyMC and Stan on the Multinomial-Dirichlet use an adjusted transform (their
default simplex transforms replaced with stick-breaking) to match the transform
used in NumPyro and TFP.
\*\*TFP: with manually added transformation when required.

PyMC, TFP and NumPyro runs are 50,000 iterations with 10 seeds, and the last
500 iterations (49,501 – 50,000) are tracked at every iteration and saved.
Stan only reports its final fit, so every tracked Stan iteration is a separate
CmdStan rerun; to keep that affordable Stan runs are 25,000 iterations with 5
seeds and the last 500 iterations (24,501 – 25,000) tracked. The plots
summarise the last `WINDOW_ITERS` (default 500) tracked iterations of every
run, and a footnote under each heatmap states the run lengths.

**Metrics.** For one seed, let $\langle\mu\rangle$ and $\langle\sigma\rangle$ be the
variational posterior mean $\mu$ and standard deviation $\sigma$ averaged over the
last `WINDOW_ITERS` iterations, and $\mu^*$, $\sigma^*$ the best variational
approximation of the posterior mean and SD.

| # | Heatmap | Definition |
|---|---|---|
| 1 | Relative bias of the VI posterior mean $\mu$ | mean over seeds of $\lvert \langle\mu\rangle - \mu^* \rvert / \lvert \mu^* \rvert$ |
| 2 | Relative bias of the VI posterior SD $\sigma$ | mean over seeds of $\lvert \langle\sigma\rangle - \sigma^* \rvert / \sigma^*$ |
| 3 | Relative between-seed noise of the VI posterior mean $\mu$ | $\mathrm{SD}_{\text{seeds}}\big(\langle\mu\rangle / \lvert \mu^* \rvert\big)$ |
| 4 | Relative between-seed noise of the VI posterior SD $\sigma$ | $\mathrm{SD}_{\text{seeds}}\big(\langle\sigma\rangle / \sigma^*\big)$ |
| 5 | Relative within-seed noise of the VI posterior mean $\mu$ | $\sqrt{\text{mean over seeds of } \mathrm{Var}_{\text{window}}\big(\mu / \lvert \mu^* \rvert\big)}$ |
| 6 | Relative within-seed noise of the VI posterior SD $\sigma$ | $\sqrt{\text{mean over seeds of } \mathrm{Var}_{\text{window}}\big(\sigma / \sigma^*\big)}$ |
| 7 | Error of the VI posterior mean $\mu$ in units of the best SD $\sigma^*$ | mean over seeds of $\lvert \langle\mu\rangle - \mu^* \rvert / \sigma^*$ |

Every metric is relative to the best variational approximation (metrics 1–6
divide by $\lvert\mu^*\rvert$ or $\sigma^*$, metric 7 by $\sigma^*$), the noise
metrics as SDs rather than variances so they are on the same scale as the
biases, and all heatmaps share one colour scale: a colour means the same value in
every plot. (`NOISE_SCALE` and `SHARED_COLOR_SCALE` in the settings cell change this.)

The bottom row, **Journé et al. example**, is the Dirichlet regression behind Table S1 of
Journé et al. (2025), run by the `--restarts` mode of the scripts in
`Perennial_reproduction/` (`pymc_reproducing.py`, `numpyro_tfp_reproducing.py`,
`stan_reproducing.py`) rather than by Part 1. It is treated as an 8-dimensional
model: its coordinates are the 8 Table S1 fixed effects, and every package is compared
with one shared best variational approximation (`Perennial_reproduction/TableS1_best_vi.csv`).
PyMC, TFP and NumPyro run 75,000 iterations for this example (Stan 25,000, tracking only every
10th of its last 500 iterations, i.e. 50 CmdStan reruns per seed, which the metrics average over), and its
footnote under each heatmap states this for the manuscript caption.

For models with more than one parameter each metric is computed per coordinate
and then combined across coordinates (the mean by default); a second set of
heatmaps shows every coordinate as its own row, with the mean, median, max and
RMS-standardized summaries underneath, so the choice of summary can be checked.

**Layout.** Part 1 runs every package × model random restart (through
`overall_bias_runners/run_all_runners.sh`, which calls each package's runner in
that package's conda environment) and saves everything to
`overall_bias_results/`. Part 2 only reads those files, so it runs in any
environment with numpy / pandas / plotly / matplotlib. Set `PLOT_ONLY = True`
to skip Part 1. Every table and heatmap function lives in
`overall_bias_runners/bias_heatmaps.py`, which `overall_default_bias_comparisons.ipynb`
uses too, so both notebooks compute and draw the metrics with the same code.

In [1]:
import os
import shlex
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# skip every cell that runs / saves random restarts and only read + plot
PLOT_ONLY = True


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "modulars").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate repo root containing modulars/")


REPO_ROOT = find_repo_root()
PAPER_VIGNETTES = REPO_ROOT / "paper_vignettes"
RUNNERS_DIR = PAPER_VIGNETTES / "overall_bias_runners"
RESULTS_DIR = Path(os.environ.get("OVERALL_BIAS_RESULTS_DIR", PAPER_VIGNETTES / "overall_bias_results")).resolve()

if str(RUNNERS_DIR) not in sys.path:
    sys.path.insert(0, str(RUNNERS_DIR))
import bias_io  # numpy / pandas only, shared with the runners
import bias_heatmaps as bh  # the tables and heatmaps, shared with overall_default_bias_comparisons.ipynb

print(f"repo root:   {REPO_ROOT}")
print(f"results dir: {RESULTS_DIR}")

## Part 1: run and save the random restarts

Every setup is the exact setup from that model's `single_MC` random-restart
notebook (see the docstrings in `overall_bias_runners/*_runner.py`), changed
only to run `RUN_MAX_ITERS` iterations and to record the last `RUN_WINDOW`
iterations at every iteration (the `track_start` option of the `modulars`
PyMC / TFP / NumPyro runners, and `min_iter` for Stan's prefix reruns).

Outputs, per model folder in `overall_bias_results/`:
`<package>_tracking.csv.gz` (one row per setup × statistic × seed × coordinate,
one column per tracked iteration), `<package>_reference.csv` (best mean / SD per
coordinate, computed as in that package's notebook) and `<package>_run_info.json`.
Finished seeds are cached under `overall_bias_results/_seed_cache/`, and models
that are already saved are skipped unless `RUN_OVERWRITE = 1`, so relaunching
resumes where it stopped.

**Run time.** These are long runs. Stan is the expensive part because CmdStan
only reports the final fit, so each of the `RUN_STAN_WINDOW` tracked Stan
iterations is a full rerun of that seed (radon with `grad_samples=100` takes
about 40 seconds per 25,000-iteration fit). That is why Stan has its own,
smaller `RUN_STAN_MAX_ITERS`, `RUN_STAN_N_SEEDS` and `RUN_STAN_WINDOW`.

In [2]:
# run settings, passed to run_all_runners.sh (each is also overridable there)
RUN_MAX_ITERS = 50_000
RUN_N_SEEDS = 10
RUN_WINDOW = 500                  # tracked final iterations (plots can use up to this many)
RUN_STAN_MAX_ITERS = 25_000       # Stan's own settings: each tracked Stan iteration is a full CmdStan rerun
RUN_STAN_N_SEEDS = 5
RUN_STAN_WINDOW = 500
RUN_OVERWRITE = 0                 # 1 reruns and overwrites saved results
RUN_PACKAGES = ["pymc", "tfp", "numpyro", "stan"]
RUN_MODELS = list(bias_io.MODEL_KEYS)
RUN_PARALLEL_PACKAGES = 1         # 1 runs the four packages at the same time
RUN_STAN_MAX_WORKERS = None       # None -> cores - 1

# "print": show the command to paste into a terminal (recommended, the runs take days)
# "background": launch it detached from this kernel, logging to overall_bias_results/logs/
# "foreground": run it inside this kernel and wait for it to finish
RUN_MODE = "print"

In [3]:
if PLOT_ONLY:
    print("PLOT_ONLY = True: skipping the random-restart runs")
else:
    run_env = {
        "MAX_ITERS": str(RUN_MAX_ITERS),
        "N_SEEDS": str(RUN_N_SEEDS),
        "WINDOW": str(RUN_WINDOW),
        "STAN_MAX_ITERS": str(RUN_STAN_MAX_ITERS),
        "STAN_N_SEEDS": str(RUN_STAN_N_SEEDS),
        "STAN_WINDOW": str(RUN_STAN_WINDOW),
        "OVERWRITE": str(RUN_OVERWRITE),
        "PACKAGES": " ".join(RUN_PACKAGES),
        "MODELS": " ".join(RUN_MODELS),
        "PARALLEL_PACKAGES": str(RUN_PARALLEL_PACKAGES),
        "RESULTS_DIR": str(RESULTS_DIR),
    }
    if RUN_STAN_MAX_WORKERS:
        run_env["STAN_MAX_WORKERS"] = str(RUN_STAN_MAX_WORKERS)
    script = RUNNERS_DIR / "run_all_runners.sh"
    command = " ".join(f"{k}={shlex.quote(v)}" for k, v in run_env.items()) + f" {shlex.quote(str(script))}"

    if RUN_MODE == "print":
        print("Run this in a terminal (any environment; it calls each package's conda env itself):\n")
        print(command)
    elif RUN_MODE == "background":
        log_dir = RESULTS_DIR / "logs"
        log_dir.mkdir(parents=True, exist_ok=True)
        log_file = open(log_dir / "run_all_runners.log", "a")
        process = subprocess.Popen(["bash", str(script)], env={**os.environ, **run_env},
                                   stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)
        print(f"launched run_all_runners.sh (pid {process.pid}); progress in {log_dir}")
    elif RUN_MODE == "foreground":
        completed = subprocess.run(["bash", str(script)], env={**os.environ, **run_env})
        print(f"run_all_runners.sh exited with code {completed.returncode}")
    else:
        raise ValueError(f"RUN_MODE must be 'print', 'background' or 'foreground', got {RUN_MODE!r}")

PLOT_ONLY = True: skipping the random-restart runs


## Part 2: read the saved results and make the heatmaps

### What has been saved so far

In [4]:
bh.saved_status(RESULTS_DIR)

### Load everything that has been saved

`RESULTS[(model, package)]` holds the windowed trajectories
(`{setup: {"mean": (seeds, iterations, coordinates), "std": ...}}`), the
tracked iteration numbers and run info of each setup, the coordinate names and
the reference values. Missing (model, package) pairs are simply left out and show up as
empty cells.

In [5]:
RESULTS = bh.load_all_results(RESULTS_DIR)
print(f"loaded {len(RESULTS)} (model, package) result sets")

### Metric definitions

`per_coordinate_metrics` (in `bias_io.py`) computes the seven metrics separately for every
coordinate (a 1D model has one coordinate). `BiasHeatmaps.combine_coordinates` (in `bias_heatmaps.py`) reduces them
to one number per heatmap cell:

- `"mean"`, `"median"`, `"max"`: that summary of the per-coordinate values
  (coordinates whose best mean is 0 are left out of metric 1).
- `"rms_standardized"`: every coordinate measured in units of its own best SD
  $\sigma^*_j$, so coordinates on different scales count equally. Bias metrics
  (1, 2, 7) become the mean over seeds of $\sqrt{\tfrac1D \sum_j e_{j}^2}$ with
  $e_j = (\langle\mu\rangle_j - \mu^*_j)/\sigma^*_j$ (metrics 1 and 7) or
  $(\langle\sigma\rangle_j - \sigma^*_j)/\sigma^*_j$ (metric 2); noise metrics (3–6), with $V_j$ the
  coordinate's variance, become $\sqrt{\tfrac1D \sum_j V_j / \sigma^{*2}_j}$
  (without the square root when `NOISE_SCALE` is not `"relative_sd"`).

### Tables behind every heatmap

`BiasHeatmaps.build_primary_tables` gives one model × setup table per metric;
`BiasHeatmaps.build_per_coord_tables` gives one coordinate × setup table per metric for a
multi-dimensional model, with the coordinate summaries as extra rows. Both read
the settings `S` from the cell below when they are called, and the heatmaps are drawn by
`BiasHeatmaps.plot_heatmap` (matplotlib, or plotly).

### Settings, then the primary heatmaps (one per metric)

Everything that controls what is computed and how it is drawn is the settings
object `S`, set up in this one cell right before the plots: the window, the row / column order and
labels, colours and colour scales (log / linear), titles, footnote marks and
the plotting backend (plotly, or matplotlib as a fallback). Change anything here
and rerun this cell and the plotting cells below.

In [9]:
# Every setting and its default is in overall_bias_runners/bias_heatmaps.py (default_settings), shared with
# overall_default_bias_comparisons.ipynb: rows / columns and their labels, titles, colours and colour scales, footnote
# marks and texts, per-coordinate heatmaps and figure output. Change any of them here, e.g. S.PLOT_BACKEND = "plotly",
# then rerun this cell and the plotting cells below. The ones changed most often:
S = bh.default_settings(RESULTS_DIR)

# ---- what to compute -----------------------------------------------------
S.WINDOW_ITERS = 500              # average / variance over the last WINDOW_ITERS iterations (<= saved window, 500)
S.PRIMARY_COORD_AGG = "mean"      # how multi-dimensional models become one cell: "mean", "median", "max", "rms_standardized"
S.NOISE_SCALE = "relative_sd"     # noise metrics (3-6): "relative_sd" = SD of (μ / |μ*|) or (σ / σ*), dividing before taking the
                                  # SD, on the same relative scale as the biases; "relative_var" = variance of those ratios; "raw" = variance

# ---- figure output ----------------------------------------------------------
S.PLOT_BACKEND = "matplotlib"     # "matplotlib" (static images, easy to copy / paste) or "plotly" (interactive)
S.NOTES_LOCATION = "console"      # "console": subtitle and footnotes printed under each plot; "figure": drawn inside it
S.SAVE_FIGURES = False            # save every heatmap to S.FIGURE_DIR (overall_bias_results/figures)
S.SAVE_TABLES = False             # also save every heatmap's numbers there as csv


In [10]:
# Primary heatmaps: one per metric. Rows are S.ROW_ORDER, columns are S.COLUMN_ORDER, and
# multi-dimensional models are combined over their coordinates with S.PRIMARY_COORD_AGG.
HM = bh.BiasHeatmaps(RESULTS, S)
PRIMARY_TABLES, PRIMARY_ZERO_FLAGS, PRIMARY_AVAILABLE = HM.build_primary_tables()
PER_COORD_TABLES = {model: HM.build_per_coord_tables(model) for model in S.PER_COORD_MODELS}
# one colour range across the primary and per-coordinate heatmaps (SHARED_COLOR_SCALE)
HM.set_shared_range([PRIMARY_TABLES] + [t[0] for t in PER_COORD_TABLES.values() if t[0] is not None])
PRIMARY_FIGURES = HM.plot_primary_heatmaps(PRIMARY_TABLES, PRIMARY_ZERO_FLAGS, PRIMARY_AVAILABLE)

### Per-coordinate heatmaps for the multi-dimensional models

One heatmap per (model, metric) in `S.PER_COORD_MODELS` × `S.PER_COORD_METRICS`:
every coordinate is its own row, followed by the summaries in
`S.PER_COORD_AGG_ROWS` (mean, median, max and RMS standardized), so the summary
used in the primary heatmaps can be compared against the alternatives.

In [11]:
PER_COORD_FIGURES = HM.plot_per_coord_heatmaps(PER_COORD_TABLES)

### Run details

The settings each saved setup was run with (from its `run_info.json`).

In [12]:
bh.run_details(RESULTS)